In [1]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   HYROX — GEOINT XGBoost Classifier  │  End-to-End Training Pipeline       ║
║                                                                              ║
║   Input  : geoint_cluster.csv  (50 000 rows)                                ║
║   Output : geoint_xgb_model.pkl  +  full evaluation report                  ║
║                                                                              ║
║   Pipeline:                                                                  ║
║     1. Load & validate data                                                  ║
║     2. Feature engineering (6 new derived features)                          ║
║     3. Stratified train / val / test split  (70 / 15 / 15)                  ║
║     4. Optuna hyperparameter search  (50 trials, pruned)                     ║
║     5. Final model training with best params + early stopping                ║
║     6. Full evaluation — accuracy, F1, ROC-AUC, confusion matrix            ║
║     7. SHAP feature importance                                               ║
║     8. Save model + scaler + metadata as .pkl                               ║
║                                                                              ║
║   Install:                                                                   ║
║     pip install xgboost scikit-learn optuna shap matplotlib seaborn          ║
║                                                                              ║
║   Run:                                                                       ║
║     python train_geoint_xgboost.py                                           ║
║     python train_geoint_xgboost.py --data path/to/geoint_cluster.csv        ║
║     python train_geoint_xgboost.py --no-optuna   (skip HPO, use defaults)  ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

'\n╔══════════════════════════════════════════════════════════════════════════════╗\n║   HYROX — GEOINT XGBoost Classifier  │  End-to-End Training Pipeline       ║\n║                                                                              ║\n║   Input  : geoint_cluster.csv  (50 000 rows)                                ║\n║   Output : geoint_xgb_model.pkl  +  full evaluation report                  ║\n║                                                                              ║\n║   Pipeline:                                                                  ║\n║     1. Load & validate data                                                  ║\n║     2. Feature engineering (6 new derived features)                          ║\n║     3. Stratified train / val / test split  (70 / 15 / 15)                  ║\n║     4. Optuna hyperparameter search  (50 trials, pruned)                     ║\n║     5. Final model training with best params + early stopping                ║\n║     6. Full eval

In [2]:

# ── stdlib ─────────────────────────────────────────────────────────────────────
import argparse, json, os, warnings, time
from datetime import datetime
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# ── third-party ────────────────────────────────────────────────────────────────
import numpy  as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")            # non-interactive backend (safe on any server)
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection    import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing      import StandardScaler, LabelEncoder
from sklearn.metrics            import (accuracy_score, f1_score, roc_auc_score,
                                        classification_report, confusion_matrix,
                                        ConfusionMatrixDisplay)
from sklearn.utils.class_weight import compute_class_weight

import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── CLI args ───────────────────────────────────────────────────────────────────
parser = argparse.ArgumentParser()

parser.add_argument("--data",      default="/Users/amitsingh/ML_Projects/WarfareAI/datasets/geoint_cluster_v2.csv")
parser.add_argument("--out",       default="models_v2")
parser.add_argument("--trials",    type=int, default=50)
parser.add_argument("--no-optuna", action="store_true")
parser.add_argument("--seed",      type=int, default=42)

# ONLY THIS (for Jupyter compatibility)
args, _ = parser.parse_known_args()


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — LOAD & VALIDATE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*64)
print("  HYROX GEOINT XGBoost — Training Pipeline")
print("═"*64)

print(f"\n[1/8] Loading data from  {args.data}")
df = pd.read_csv(args.data)
print(f"      Shape: {df.shape}   |   Columns: {df.columns.tolist()}")
print(f"      Label distribution:\n{df.threat_label.value_counts().to_string()}")
assert df.isnull().sum().sum() == 0, "Dataset contains nulls — fix before training"
print("      ✓ No missing values")



════════════════════════════════════════════════════════════════
  HYROX GEOINT XGBoost — Training Pipeline
════════════════════════════════════════════════════════════════

[1/8] Loading data from  /Users/amitsingh/ML_Projects/WarfareAI/datasets/geoint_cluster_v2.csv
      Shape: (52480, 23)   |   Columns: ['cluster_id', 'timestamp', 'region', 'n_tanks', 'n_troops', 'n_drones', 'n_vehicles', 'n_total_units', 'mean_velocity_kmh', 'max_velocity_kmh', 'velocity_std', 'cluster_radius_km', 'dist_from_border_km', 'units_in_5km_strip', 'heavy_armor_ratio', 'air_ratio', 'n_active_scenarios', 'unit_density', 'velocity_pressure', 'proximity_factor', 'mobility_index', 'threat_label', 'threat_label_int']
      Label distribution:
threat_label
LOW         17954
MODERATE    15620
HIGH        11358
CRITICAL     7548
      ✓ No missing values


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — FEATURE ENGINEERING
#
# Base features (13, directly from simulator payload):
#   n_tanks, n_troops, n_drones, n_vehicles, n_total_units
#   mean_velocity_kmh, max_velocity_kmh, velocity_std
#   cluster_radius_km, dist_from_border_km, units_in_5km_strip
#   heavy_armor_ratio, air_ratio, n_active_scenarios
#
# Engineered features (6 new):
#   speed_border_index    — velocity × proximity (high speed + close = danger)
#   armor_border_index    — heavy_armor_ratio × proximity
#   total_threat_signal   — mirrors simulator's raw score: geo×1.2 + scenarios×8
#   velocity_spread       — vel_std / (mean_vel + 1) — measures formation chaos
#   strip_density         — units_in_5km_strip / (n_total + 1)
#   scenario_multiplier   — log1p(n_active_scenarios)  (log dampens extremes)
# ══════════════════════════════════════════════════════════════════════════════
print("\n[2/8] Feature engineering …")

# proximity 0→1 (1 = on border, 0 = far away)
# dist_from_border_km ranges roughly 0–400 across all regions
df["proximity_score"]     = 1.0 / (1.0 + df["dist_from_border_km"] / 50.0)
df["speed_border_index"]  = df["mean_velocity_kmh"] * df["proximity_score"]
df["armor_border_index"]  = df["heavy_armor_ratio"]  * df["proximity_score"]
df["total_threat_signal"] = df["n_total_units"] * 1.2 + df["n_active_scenarios"] * 8.0
df["velocity_spread"]     = df["velocity_std"] / (df["mean_velocity_kmh"] + 1.0)
df["strip_density"]       = df["units_in_5km_strip"] / (df["n_total_units"] + 1.0)
df["scenario_multiplier"] = np.log1p(df["n_active_scenarios"])

FEATURE_COLS = [
    # raw counts
    "n_tanks", "n_troops", "n_drones", "n_vehicles", "n_total_units",
    # velocity
    "mean_velocity_kmh", "max_velocity_kmh", "velocity_std",
    # spatial
    "cluster_radius_km", "dist_from_border_km", "units_in_5km_strip",
    # ratios
    "heavy_armor_ratio", "air_ratio",
    # scenario
    "n_active_scenarios",
    # engineered
    "proximity_score", "speed_border_index", "armor_border_index",
    "total_threat_signal", "velocity_spread", "strip_density",
    "scenario_multiplier",
]

print(f"      Total features: {len(FEATURE_COLS)}")
print(f"      New engineered: proximity_score, speed_border_index, armor_border_index,")
print(f"                       total_threat_signal, velocity_spread, strip_density,")
print(f"                       scenario_multiplier")

X = df[FEATURE_COLS].values.astype(np.float32)
LABEL_INT = {'LOW': 0, 'MODERATE': 1, 'HIGH': 2}
y = df["threat_label"].map(LABEL_INT).values.astype(np.int32)


[2/8] Feature engineering …
      Total features: 21
      New engineered: proximity_score, speed_border_index, armor_border_index,
                       total_threat_signal, velocity_spread, strip_density,
                       scenario_multiplier


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — STRATIFIED SPLIT  (70 / 15 / 15)
# ══════════════════════════════════════════════════════════════════════════════
print("\n[3/8] Stratified split  70 / 15 / 15 …")

RNG = args.seed
LABEL_ORDER = list(LABEL_INT.keys())

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RNG, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15/0.85,
    random_state=RNG, stratify=y_train_val)

print(f"      Train : {len(X_train):,}  |  Val : {len(X_val):,}  |  Test : {len(X_test):,}")

# Scale features
scaler  = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# DMatrix for XGBoost native API
dtrain = xgb.DMatrix(X_train_s, label=y_train, feature_names=FEATURE_COLS)
dval   = xgb.DMatrix(X_val_s,   label=y_val,   feature_names=FEATURE_COLS)
dtest  = xgb.DMatrix(X_test_s,  label=y_test,  feature_names=FEATURE_COLS)

# Class weights to handle CRITICAL imbalance
class_counts = np.bincount(y_train)
class_weights = len(y_train) / (len(LABEL_ORDER) * class_counts)
sample_weights = class_weights[y_train]
print(f"      Class weights: {dict(zip(LABEL_ORDER, class_weights.round(3)))}")
 


[3/8] Stratified split  70 / 15 / 15 …
      Train : 36,735  |  Val : 7,873  |  Test : 7,872
      Class weights: {'LOW': np.float64(0.686), 'MODERATE': np.float64(1.12), 'HIGH': np.float64(1.54)}


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — OPTUNA HYPERPARAMETER SEARCH
#
#  Search space (all ranges chosen around XGBoost best-practice literature):
#    n_estimators         200 – 1000   (more trees = better, capped by early stop)
#    max_depth            3 – 9        (4-6 best for tabular; 3=underfit, 9=overfit)
#    learning_rate        0.01 – 0.30  (log scale; lower = needs more trees)
#    subsample            0.60 – 1.00  (row sampling per tree; 0.8 is classic)
#    colsample_bytree     0.50 – 1.00  (feature sampling per tree)
#    colsample_bylevel    0.50 – 1.00  (feature sampling per split level)
#    min_child_weight     1 – 20       (min sum of weights in leaf; fights CRITICAL overfit)
#    gamma               0.0 – 5.0    (min loss reduction for split; 0 = no pruning)
#    reg_alpha            0.0 – 2.0    (L1 on leaf weights)
#    reg_lambda           0.5 – 5.0    (L2 on leaf weights; always > 0)
#    scale_pos_weight     auto         (handles class imbalance per class implicitly)
#    max_delta_step       0 – 5        (stabilises leaf output — helps with imbalance)
# ══════════════════════════════════════════════════════════════════════════════

if args.no_optuna:
    print("\n[4/8] Skipping Optuna — using default hyperparameters …")
    best_params = dict(
        n_estimators=600, max_depth=6, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.80, colsample_bylevel=0.80,
        min_child_weight=3, gamma=0.1, reg_alpha=0.2, reg_lambda=2.0,
        max_delta_step=1,
    )
else:
    print(f"\n[4/8] Optuna hyperparameter search  ({args.trials} trials) …")
    t0 = time.time()

    def objective(trial):
        params = dict(
            objective         = "multi:softprob",
            num_class         = len(LABEL_ORDER),
            eval_metric       = ["mlogloss", "merror"],
            tree_method       = "hist",
            device            = "cpu",
            n_estimators      = trial.suggest_int("n_estimators", 200, 1000),
            max_depth         = trial.suggest_int("max_depth", 3, 9),
            learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
            subsample         = trial.suggest_float("subsample", 0.60, 1.00),
            colsample_bytree  = trial.suggest_float("colsample_bytree", 0.50, 1.00),
            colsample_bylevel = trial.suggest_float("colsample_bylevel", 0.50, 1.00),
            min_child_weight  = trial.suggest_int("min_child_weight", 1, 20),
            gamma             = trial.suggest_float("gamma", 0.0, 5.0),
            reg_alpha         = trial.suggest_float("reg_alpha", 0.0, 2.0),
            reg_lambda        = trial.suggest_float("reg_lambda", 0.5, 5.0),
            max_delta_step    = trial.suggest_int("max_delta_step", 0, 5),
            seed              = RNG,
            verbosity         = 0,
        )
        evals_result = {}
        bst = xgb.train(
            params,
            dtrain,
            num_boost_round = params.pop("n_estimators"),
            evals           = [(dval, "validation")],
            early_stopping_rounds = 30,
            evals_result    = evals_result,
            verbose_eval    = False,
        )
        y_pred = bst.predict(dval).reshape(-1, len(LABEL_ORDER)).argmax(axis=1)
        return accuracy_score(y_val, y_pred)

    sampler = optuna.samplers.TPESampler(seed=RNG, n_startup_trials=10)
    pruner  = optuna.pruners.MedianPruner(n_warmup_steps=10)
    study   = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
    study.optimize(objective, n_trials=args.trials, show_progress_bar=False)

    best_params = study.best_params
    print(f"      Search complete in {time.time()-t0:.1f}s")
    print(f"      Best val accuracy : {study.best_value:.4f}")
    print(f"      Best params:")
    for k, v in best_params.items():
        print(f"        {k:<25} {v}")



[4/8] Optuna hyperparameter search  (50 trials) …
      Search complete in 7.7s
      Best val accuracy : 0.4862
      Best params:
        n_estimators              525
        max_depth                 7
        learning_rate             0.0318476763961877
        subsample                 0.8614838463376575
        colsample_bytree          0.628353469484714
        colsample_bylevel         0.7048228942714392
        min_child_weight          13
        gamma                     3.026511841185572
        reg_alpha                 0.3666107116281516
        reg_lambda                4.49990186723194
        max_delta_step            1


In [7]:
# STEP 5 — FINAL MODEL TRAINING
#   Uses XGBoost native API (xgb.train) with early stopping on val set.
#   Trains on TRAIN+VAL combined after HPO is done — maximises data usage.
# ══════════════════════════════════════════════════════════════════════════════
print("\n[5/8] Training final model …")

final_params = dict(
    objective         = "multi:softprob",
    num_class         = len(LABEL_ORDER),
    eval_metric       = ["mlogloss", "merror"],
    tree_method       = "hist",
    device            = "cpu",
    max_depth         = best_params.get("max_depth", 6),
    learning_rate     = best_params.get("learning_rate", 0.05),
    subsample         = best_params.get("subsample", 0.85),
    colsample_bytree  = best_params.get("colsample_bytree", 0.80),
    colsample_bylevel = best_params.get("colsample_bylevel", 0.80),
    min_child_weight  = best_params.get("min_child_weight", 3),
    gamma             = best_params.get("gamma", 0.1),
    reg_alpha         = best_params.get("reg_alpha", 0.2),
    reg_lambda        = best_params.get("reg_lambda", 2.0),
    max_delta_step    = best_params.get("max_delta_step", 1),
    seed              = RNG,
    verbosity         = 0,
)
n_est = best_params.get("n_estimators", 600)

evals_result = {}
t0 = time.time()

# Phase A: train on TRAIN, validate on VAL to get best iteration
bst_phase_a = xgb.train(
    final_params,
    dtrain,
    num_boost_round       = n_est,
    evals                 = [(dtrain, "train"), (dval, "val")],
    early_stopping_rounds = 40,
    evals_result          = evals_result,
    verbose_eval          = 50,
)
best_iter = bst_phase_a.best_iteration
print(f"\n      Best iteration from early stopping: {best_iter}")

# Phase B: retrain on TRAIN+VAL combined for exactly best_iter rounds
X_full_s = scaler.transform(np.vstack([X_train, X_val]))
y_full   = np.concatenate([y_train, y_val])
dfull    = xgb.DMatrix(X_full_s, label=y_full, feature_names=FEATURE_COLS)

final_params_copy = {k: v for k, v in final_params.items()
                     if k not in ("eval_metric",)}
final_params_copy["eval_metric"] = "mlogloss"

model = xgb.train(
    final_params_copy,
    dfull,
    num_boost_round = best_iter + 1,
    verbose_eval    = False,
)
print(f"      Training time: {time.time()-t0:.1f}s  |  Trees: {best_iter+1}")




[5/8] Training final model …
[0]	train-mlogloss:1.09441	train-merror:0.51112	val-mlogloss:1.09461	val-merror:0.51683
[45]	train-mlogloss:1.00478	train-merror:0.51379	val-mlogloss:1.01299	val-merror:0.51391

      Best iteration from early stopping: 5
      Training time: 0.3s  |  Trees: 6


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — EVALUATION
# ══════════════════════════════════════════════════════════════════════════════
print("\n[6/8] Evaluation on held-out test set …")

prob_test  = model.predict(dtest).reshape(-1, len(LABEL_ORDER))
y_pred     = prob_test.argmax(axis=1)

acc        = accuracy_score(y_test, y_pred)
f1_macro   = f1_score(y_test, y_pred, average="macro")
f1_weighted= f1_score(y_test, y_pred, average="weighted")

# Per-class ROC-AUC (one-vs-rest)
y_test_bin = np.eye(len(LABEL_ORDER))[y_test]
try:
    roc_auc = roc_auc_score(y_test_bin, prob_test, multi_class="ovr", average="macro")
except Exception:
    roc_auc = float("nan")

print(f"\n  ┌─────────────────────────────────────────┐")
print(f"  │  Test Accuracy       : {acc:.4f}           │")
print(f"  │  Macro F1            : {f1_macro:.4f}           │")
print(f"  │  Weighted F1         : {f1_weighted:.4f}           │")
print(f"  │  ROC-AUC (macro OvR) : {roc_auc:.4f}           │")
print(f"  └─────────────────────────────────────────┘")

print("\n  Per-class report:")
print(classification_report(y_test, y_pred,
                             target_names=LABEL_ORDER, digits=4))

# 5-fold cross-validation on full dataset (sanity check)
print("  5-fold CV on full dataset (XGBClassifier wrapper) …")
from xgboost import XGBClassifier
cv_model = XGBClassifier(**{k: v for k, v in final_params.items()
                             if k not in ("objective","num_class","eval_metric",
                                          "tree_method","device","verbosity")},
                          objective="multi:softprob", num_class=len(LABEL_ORDER),
                          n_estimators=best_iter+1, use_label_encoder=False,
                          tree_method="hist", verbosity=0, random_state=RNG)
cv_scores = cross_val_score(cv_model, scaler.transform(X), y,
                             cv=StratifiedKFold(5, shuffle=True, random_state=RNG),
                             scoring="accuracy", n_jobs=-1)
print(f"  CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  "
      f"(folds: {[f'{s:.4f}' for s in cv_scores]})")

# Confusion matrix plot
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=LABEL_ORDER)
disp.plot(ax=ax, colorbar=False, cmap="Blues", values_format="d")
ax.set_title("GEOINT XGBoost — Confusion Matrix (Test Set)")
plt.tight_layout()
cm_path = os.path.join(args.out, "geoint_confusion_matrix.png")
os.makedirs(os.path.dirname(cm_path), exist_ok=True)
plt.savefig(cm_path, dpi=150)
plt.close()
print(f"\n  Confusion matrix saved → {cm_path}")



[6/8] Evaluation on held-out test set …

  ┌─────────────────────────────────────────┐
  │  Test Accuracy       : 0.4859           │
  │  Macro F1            : 0.2197           │
  │  Weighted F1         : 0.3191           │
  │  ROC-AUC (macro OvR) : 0.6175           │
  └─────────────────────────────────────────┘

  Per-class report:
              precision    recall  f1-score   support

         LOW     0.4863    0.9987    0.6541      3825
    MODERATE     0.3000    0.0013    0.0025      2343
        HIGH     0.2857    0.0012    0.0023      1704

    accuracy                         0.4859      7872
   macro avg     0.3573    0.3337    0.2197      7872
weighted avg     0.3874    0.4859    0.3191      7872

  5-fold CV on full dataset (XGBClassifier wrapper) …
  CV Accuracy: 0.4857 ± 0.0008  (folds: ['0.4859', '0.4871', '0.4853', '0.4846', '0.4858'])

  Confusion matrix saved → models_v2/geoint_confusion_matrix.png


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 — SHAP FEATURE IMPORTANCE
# ══════════════════════════════════════════════════════════════════════════════
print("\n[7/8] SHAP feature importance …")
try:
    import shap
    explainer   = shap.TreeExplainer(model)
    shap_sample = X_test_s[:2000]
    shap_dmat   = xgb.DMatrix(shap_sample, feature_names=FEATURE_COLS)
    shap_values = explainer.shap_values(shap_sample)
    sv = np.array(shap_values)
    # shape is (n_samples, n_features, n_classes) for XGBoost multiclass
    if sv.ndim == 3 and sv.shape[-1] == len(LABEL_ORDER):
        mean_abs_shap = np.abs(sv).mean(axis=(0, 2))
    elif sv.ndim == 3:
        mean_abs_shap = np.abs(sv).mean(axis=(0, 1))
    elif isinstance(shap_values, list):
        mean_abs_shap = np.mean([np.abs(np.array(s)).mean(axis=0) for s in shap_values], axis=0)
    else:
        mean_abs_shap = np.abs(sv).mean(axis=0)
    mean_abs_shap = np.array(mean_abs_shap).flatten()[:len(FEATURE_COLS)]
    shap_df = pd.DataFrame({
        "feature":       FEATURE_COLS,
        "mean_abs_shap": mean_abs_shap,
    }).sort_values("mean_abs_shap", ascending=True)

    fig, ax = plt.subplots(figsize=(7, 7))
    colors = ["#0F6E56" if "engineered" not in f else "#185FA5"
              for f in shap_df["feature"]]
    bars = ax.barh(shap_df["feature"], shap_df["mean_abs_shap"], color="#0F6E56", alpha=0.85)
    # highlight engineered features
    for i, feat in enumerate(shap_df["feature"]):
        if feat in ["proximity_score","speed_border_index","armor_border_index",
                    "total_threat_signal","velocity_spread","strip_density",
                    "scenario_multiplier"]:
            bars[i].set_color("#185FA5")
            bars[i].set_alpha(0.85)
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title("GEOINT XGBoost — Feature Importance (SHAP)\n"
                 "Green = raw simulator feature   Blue = engineered feature")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    shap_path = os.path.join(args.out, "geoint_shap_importance.png")
    plt.savefig(shap_path, dpi=150)
    plt.close()
    print(f"  SHAP plot saved → {shap_path}")
    print("\n  Top 10 features by |SHAP|:")
    for _, row in shap_df.tail(10).iloc[::-1].iterrows():
        tag = " ← engineered" if row.feature in [
            "proximity_score","speed_border_index","armor_border_index",
            "total_threat_signal","velocity_spread","strip_density","scenario_multiplier"
        ] else ""
        print(f"    {row.feature:<28} {row.mean_abs_shap:.5f}{tag}")

except ImportError:
    print("  shap not installed — skipping (pip install shap)")

# Training curve (from Phase A evals)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in [
    (axes[0], "mlogloss", "Log Loss"),
    (axes[1], "merror",   "Classification Error"),
]:
    if metric in evals_result.get("train", {}):
        axes[0].plot(evals_result["train"][metric], label="Train", color="#0F6E56")
    if metric in evals_result.get("val", {}):
        axes[0 if metric=="mlogloss" else 1].plot(
            evals_result["val"][metric], label="Val", color="#185FA5", linestyle="--")
    ax.set_xlabel("Boosting Round")
    ax.set_title(f"GEOINT XGBoost — {title}")
    ax.legend()
    ax.grid(alpha=0.3)

for metric, ax in [("mlogloss", axes[0]), ("merror", axes[1])]:
    if "train" in evals_result and metric in evals_result["train"]:
        ax.plot(evals_result["train"][metric], label="Train", color="#0F6E56")
    if "val" in evals_result and metric in evals_result["val"]:
        ax.plot(evals_result["val"][metric],   label="Val",   color="#D85A30", linestyle="--")
    ax.set_xlabel("Boosting Round")
    ax.set_title(f"GEOINT — {metric}")
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
curve_path = os.path.join(args.out, "geoint_training_curves.png")
plt.savefig(curve_path, dpi=150)
plt.close()
print(f"\n  Training curves saved → {curve_path}")



[7/8] SHAP feature importance …
  SHAP plot saved → models_v2/geoint_shap_importance.png

  Top 10 features by |SHAP|:
    armor_border_index           0.00905 ← engineered
    proximity_score              0.00844 ← engineered
    dist_from_border_km          0.00775
    speed_border_index           0.00383 ← engineered
    mean_velocity_kmh            0.00223
    heavy_armor_ratio            0.00206
    total_threat_signal          0.00183 ← engineered
    n_tanks                      0.00180
    units_in_5km_strip           0.00175
    cluster_radius_km            0.00146

  Training curves saved → models_v2/geoint_training_curves.png


In [10]:

# ══════════════════════════════════════════════════════════════════════════════
print("\n[8/8] Saving model bundle …")

bundle = {
    # the trained XGBoost Booster (native)
    "model":            model,
    # the fitted StandardScaler
    "scaler":           scaler,
    # exact feature list — must match backend extraction order
    "feature_cols":     FEATURE_COLS,
    # class order for argmax → label
    "label_order":      LABEL_ORDER,
    "label_int_map":    LABEL_INT,
    # metadata for the backend
    "metadata": {
        "trained_at":       datetime.now().isoformat(),
        "dataset":          args.data,
        "n_train":          len(X_train),
        "n_val":            len(X_val),
        "n_test":           len(X_test),
        "n_features":       len(FEATURE_COLS),
        "best_iteration":   best_iter,
        "best_params":      best_params,
        "test_accuracy":    round(acc, 6),
        "test_f1_macro":    round(f1_macro, 6),
        "test_f1_weighted": round(f1_weighted, 6),
        "test_roc_auc":     round(roc_auc, 6),
        "cv_mean":          round(float(cv_scores.mean()), 6),
        "cv_std":           round(float(cv_scores.std()), 6),
        "xgboost_version":  xgb.__version__,
    },
}

model_path = os.path.join(args.out, "geoint_xgb_model.pkl")
joblib.dump(bundle, model_path, compress=3)
print(f"  Model bundle saved → {model_path}")

# Also save a human-readable JSON summary
summary = bundle["metadata"].copy()
summary["confusion_matrix"] = cm.tolist()
summary_path = os.path.join(args.out, "geoint_training_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"  Training summary  → {summary_path}")

print("\n" + "═"*64)
print("  TRAINING COMPLETE")
print(f"  Test Accuracy : {acc:.4f}")
print(f"  Macro F1      : {f1_macro:.4f}")
print(f"  ROC-AUC       : {roc_auc:.4f}")
print(f"  CV (5-fold)   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print("═"*64)
print()
print("  Load in backend:")
print(f"    import joblib")
print(f"    import xgboost as xgb")
print(f"    import numpy as np")
print(f"    bundle = joblib.load('{model_path}')")
print(f"    model, scaler = bundle['model'], bundle['scaler']")
print(f"    features = bundle['feature_cols']  # keep this order")
print(f"    # inference:")
print(f"    X = scaler.transform([your_feature_vector])  # shape (1, {len(FEATURE_COLS)})")
print(f"    dmat = xgb.DMatrix(X, feature_names=features)")
print(f"    probs = model.predict(dmat).reshape(-1, 4)   # [LOW, MOD, HIGH, CRIT]")
print(f"    label = bundle['label_order'][probs.argmax(axis=1)[0]]")
print(f"    score = float(probs[0] @ [0.0, 0.35, 0.68, 0.92])  # weighted score 0-1")


[8/8] Saving model bundle …
  Model bundle saved → models_v2/geoint_xgb_model.pkl
  Training summary  → models_v2/geoint_training_summary.json

════════════════════════════════════════════════════════════════
  TRAINING COMPLETE
  Test Accuracy : 0.4859
  Macro F1      : 0.2197
  ROC-AUC       : 0.6175
  CV (5-fold)   : 0.4857 ± 0.0008
════════════════════════════════════════════════════════════════

  Load in backend:
    import joblib
    import xgboost as xgb
    import numpy as np
    bundle = joblib.load('models_v2/geoint_xgb_model.pkl')
    model, scaler = bundle['model'], bundle['scaler']
    features = bundle['feature_cols']  # keep this order
    # inference:
    X = scaler.transform([your_feature_vector])  # shape (1, 21)
    dmat = xgb.DMatrix(X, feature_names=features)
    probs = model.predict(dmat).reshape(-1, 4)   # [LOW, MOD, HIGH, CRIT]
    label = bundle['label_order'][probs.argmax(axis=1)[0]]
    score = float(probs[0] @ [0.0, 0.35, 0.68, 0.92])  # weighted score 0-